# Lab 10 · Đảm bảo chất lượng chéo bảng

**Lập trình xử lý dữ liệu (LTXLDL) · 2627-1 · Giờ thực hành · Bài 10**

> 💡 File → **Save a copy in Drive** trước khi sửa.

Notebook demo bài 10 làm sạch bảng `listings`. Lab này kiểm tra tính nhất quán giữa
hai bảng thông qua khoá ngoại, khoá tự nhiên và các cột dẫn xuất có sẵn.

*Từ tuần này phần kỹ năng kéo dài khoảng 50 phút; 30 phút cuối dành cho hoạt động hỗ trợ bài tập lớn.*

**Mục tiêu bài lab**

1. Kiểm tra miền thời gian và khoá ngoại giữa hai bảng, rồi diễn giải đúng kết quả của phép kiểm.
2. Chứng minh một bảng **không có khoá tự nhiên** và nói được hệ quả.
3. Đối chiếu cột dẫn xuất có sẵn với con số tự đếm và phân tích phần chênh lệch.
4. Đóng gói mọi phép kiểm thành `qa_report` ghi ra file.

Phiên bản đề: `qa-cross-table-v1` (25/09/2026). Lưu đúng tên **`lab-10.ipynb`**.

## Cách làm và nộp bài

- Các bài có hướng dẫn: nên **tự gõ, không dùng AI** — các bài kiểm tra định kỳ 🚫 đóng ở giờ lý thuyết
  đánh giá các kỹ năng này. Bài tự làm cuối lab được gắn nhãn ✅ mở: được dùng AI, kèm trách nhiệm khai báo.
- Điền thân hàm TODO và phần trả lời Markdown. Mỗi phép kiểm là **một hàm nhận bảng qua tham số**,
  để cùng bộ quy tắc chạy được trên mốc chụp khác (đúng yêu cầu của bài tập lớn).
- Giữ tên hàm, tham số, cell ID và metadata; viết helper/import thêm trong chính cell bài làm.
  Grader chỉ chạy cell câu đó với bảng riêng (mốc chụp khác, có review mồ côi, có cột lệch…),
  không phụ thuộc biến `ds`, `rv`, `MOC` ở cell khác. `pd` và `np` được cung cấp sẵn khi chấm.
- Public checks chỉ là ví dụ tự kiểm tra trên dữ liệu giả lập. Một check lỗi không dừng các check sau.
  TODO báo **CHƯA LÀM**; lỗi cú pháp vẫn cần sửa.
- Phép kiểm **chỉ đếm và báo cáo**: không sửa, không xoá dòng của bảng đầu vào; không ghi cứng kết quả.
- Lab chạy với hành vi **Copy-on-Write** của pandas 3 (bật sẵn trong cell dữ liệu và khi chấm).
- **Restart & Run all**, lưu notebook, commit/push vào repo của bạn trong org và tạo tag theo cuối bài.

## Rubric dự kiến — 100 điểm

| Mục | Điểm | Cách đánh giá |
|---|---:|---|
| Bài 1, 2, 3 — mỗi bài 10 điểm | 30 | Tự động |
| Bài 4 (đối chiếu tổng review) 15 · Bài 5 (lệch LTM) 20 · Bài 6 (`qa_report`) 15 | 50 | Tự động |
| Diễn giải: chính sách mốc, số 0 vi phạm, khoá tự nhiên, phân bố độ lệch | 15 | Giảng viên |
| Khai báo AI và bằng chứng kiểm chứng | 5 | Giảng viên |

Phần 🧭 hỗ trợ bài tập lớn ở cuối lab làm theo nhóm, không tính vào điểm lab.

In [ ]:
# Public checks giúp tự kiểm tra; đây không phải điểm chính thức.
# Không sửa cell này trong bài nộp.
PUBLIC_RESULTS = {}

def public_check(case_id, check):
    try:
        check()
    except NotImplementedError:
        status, detail = "CHƯA LÀM", "Điền phần TODO rồi chạy lại."
    except AssertionError as exc:
        status, detail = "CHƯA ĐẠT", str(exc) or "Kết quả chưa khớp ví dụ công khai."
    except Exception as exc:
        status, detail = "LỖI", f"{type(exc).__name__}: {exc}"
    else:
        status, detail = "ĐẠT", "Ví dụ công khai đã qua; chưa đại diện toàn bộ rubric."
    PUBLIC_RESULTS[case_id] = status
    print(f"[{status}] {case_id}: {detail}")

def require_answer(value):
    if value is None or value is Ellipsis:
        raise NotImplementedError

def preview(label, action):
    try:
        value = action()
    except NotImplementedError:
        print(f"{label}: chưa chạy được vì còn TODO.")
    except Exception as exc:
        print(f"{label}: {type(exc).__name__}: {exc}")
    else:
        print(label)
        print(value)

def show_public_summary():
    print("PUBLIC CHECKS — không phải điểm chính thức")
    for case_id, status in PUBLIC_RESULTS.items():
        print(f"{case_id}: {status}")
    print("Sau khi sửa, Restart & Run all để làm mới toàn bộ kết quả.")

## Dữ liệu dùng trong lab

Mặc định `USE_SNAPSHOT = False`: notebook dùng **dữ liệu giả lập** (6 chỗ ở, 18 đánh giá) có cùng các hiện tượng
như snapshot thật — vài đánh giá sau mốc chụp, cặp `(listing_id, date)` lặp lại, cột LTM lệch −1.

Đặt `USE_SNAPSHOT = True` để đọc bản **visualisations** (`listings.csv`, `reviews.csv`) của snapshot Santiago **2026-06-29**.
Hằng `MOC` là mốc chụp công bố; `LTM_BAT_DAU` là ngày bắt đầu cửa sổ 12 tháng thử nghiệm. Public checks luôn dùng dữ liệu giả lập.

In [ ]:
import numpy as np
import pandas as pd

# Hành vi pandas 3 (Copy-on-Write).
if pd.__version__.startswith("2."):
    pd.set_option("mode.copy_on_write", True)

BASE = "https://data.insideairbnb.com/chile/rm/santiago/2026-06-29/visualisations"
MOC = "2026-06-29"
LTM_BAT_DAU = "2025-06-30"

# Dữ liệu GIẢ LẬP: 6 chỗ ở và 18 đánh giá, cùng cấu trúc bản visualisations của Inside Airbnb.
DEMO_RV = pd.DataFrame({
    "listing_id": [11, 11, 11, 11, 11, 11, 11, 12, 12, 12, 12, 13, 13, 13, 13, 14, 14, 16],
    "date": pd.to_datetime([
        "2019-03-01", "2024-12-01", "2025-06-30", "2025-08-10", "2025-08-10", "2026-06-29", "2026-06-30",
        "2010-11-13", "2025-06-30", "2026-01-05", "2026-07-01",
        "2023-04-04", "2023-04-04", "2023-04-04", "2025-07-01",
        "2025-12-24", "2026-06-30", "2021-02-02"]),
})
DEMO_DS = pd.DataFrame({
    "id": [11, 12, 13, 14, 15, 16],
    "name": ["Depto Lastarria", "Pieza Bellavista", "Loft Providencia", "Casa Ñuñoa", "Studio nuevo", "Hab Centro"],
    "number_of_reviews": [7, 4, 4, 2, 0, 1],
    "number_of_reviews_ltm": [4, 2, 1, 2, 0, 0],
    "last_review": pd.to_datetime(["2026-06-30", "2026-07-01", "2025-07-01", "2026-06-30", None, "2021-02-02"]),
})

# Đặt USE_SNAPSHOT = True để làm trên snapshot Santiago thật.
USE_SNAPSHOT = False

if USE_SNAPSHOT:
    ds = pd.read_csv(f"{BASE}/listings.csv", parse_dates=["last_review"])
    rv = pd.read_csv(f"{BASE}/reviews.csv", parse_dates=["date"])
    DATA_LABEL = "Snapshot Santiago 2026-06-29"
else:
    ds, rv = DEMO_DS.copy(), DEMO_RV.copy()
    DATA_LABEL = "GIẢ LẬP — 6 chỗ ở, 18 đánh giá"

print(DATA_LABEL, "·", len(ds), "chỗ ở ·", len(rv), "đánh giá")

## Bài 1 · Miền thời gian của bảng đánh giá

Tên thư mục ghi mốc chụp **29/06/2026**. Các đánh giá có ngày sau mốc đó cần được gắn cờ.

### Hợp đồng hàm · 10 điểm tự động

`time_domain(rv: pd.DataFrame, moc: str) -> dict`

Nhận bảng đánh giá (cột `date` kiểu `datetime64`, ít nhất một dòng) và mốc `"YYYY-MM-DD"`. Trả dict có đúng các khóa:
`ngay_dau`, `ngay_cuoi` (`Timestamp` sớm nhất/muộn nhất của `date`) và `so_sau_moc` (int, số dòng có `date > moc`).
Không sửa input. Đối chiếu snapshot: từ **2010-11-13** đến **2026-07-01**; **204** dòng sau mốc chụp.

In [ ]:
def time_domain(rv: pd.DataFrame, moc: str) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q1():
    result = time_domain(DEMO_RV, MOC)
    assert set(result) == {"ngay_dau", "ngay_cuoi", "so_sau_moc"}
    assert pd.Timestamp(result["ngay_dau"]) == pd.Timestamp("2010-11-13")
    assert pd.Timestamp(result["ngay_cuoi"]) == pd.Timestamp("2026-07-01")
    assert result["so_sau_moc"] == 3, "30/06 (2 dòng) và 01/07 (1 dòng) nằm sau mốc 29/06"
    assert time_domain(DEMO_RV, "2026-07-01")["so_sau_moc"] == 0

public_check("Q1 · ví dụ công khai", check_q1)

**Chính sách mốc chụp:** các dòng sau mốc rơi vào những ngày nào? Có nên đổi mốc công bố thành ngày muộn nhất trong tệp không? Vì sao? …

## Bài 2 · Khoá ngoại: đánh giá không khớp chỗ ở

Mọi `listing_id` trong bảng đánh giá phải tồn tại trong `ds["id"]`. Nếu không, hai bảng không bảo toàn khoá ngoại.

### Hợp đồng hàm · 10 điểm tự động

`orphan_reviews(rv: pd.DataFrame, ds: pd.DataFrame) -> int`

Nhận bảng đánh giá (cột `listing_id`) và bảng chỗ ở (cột `id`). Trả **int**: số dòng đánh giá có `listing_id`
**không** nằm trong `ds["id"]` (dùng `isin` và `~`). Mỗi dòng mồ côi đều được đếm, kể cả khi cùng một `listing_id`
lặp lại. Không sửa input. Đối chiếu snapshot: **0** — khoá ngoại toàn vẹn.

In [ ]:
def orphan_reviews(rv: pd.DataFrame, ds: pd.DataFrame) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q2():
    result = orphan_reviews(DEMO_RV, DEMO_DS)
    assert isinstance(result, (int, np.integer)) and not isinstance(result, bool)
    assert result == 0
    thieu = DEMO_DS[DEMO_DS["id"] != 11]
    assert orphan_reviews(DEMO_RV, thieu) == 7, "Bỏ chỗ ở 11 thì 7 đánh giá của nó thành mồ côi"

public_check("Q2 · ví dụ công khai", check_q2)

**Số 0 vi phạm:** vì sao phép kiểm cho kết quả 0 vẫn phải ghi trong báo cáo? …

## Bài 3 · Khoá tự nhiên: bảng đánh giá rút gọn có khoá không?

### Hợp đồng hàm · 10 điểm tự động

`duplicate_pairs(rv: pd.DataFrame, cot: list = ["listing_id", "date"]) -> int`

Nhận bảng và danh sách cột ứng viên làm khoá. Trả **int**: số dòng lặp lại theo các cột đó
(`duplicated(subset=cot)` — lần xuất hiện đầu tiên không tính). Chỉ đếm, **không** `drop_duplicates`, không sửa input.
Đối chiếu snapshot: **3.548** dòng "trùng" theo `(listing_id, date)`.

In [ ]:
def duplicate_pairs(rv: pd.DataFrame, cot: list = ["listing_id", "date"]) -> int:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q3():
    result = duplicate_pairs(DEMO_RV)
    assert isinstance(result, (int, np.integer)) and not isinstance(result, bool)
    assert result == 3, "Chỗ ở 11 ngày 10/08/2025 (1 dòng lặp) và chỗ ở 13 ngày 04/04/2023 (2 dòng lặp)"
    assert duplicate_pairs(DEMO_RV, ["listing_id"]) == 13
    assert len(DEMO_RV) == 18

public_check("Q3 · ví dụ công khai", check_q3)

**Khoá tự nhiên:** các dòng "trùng" này có phải lỗi không? Nếu `drop_duplicates(subset=["listing_id", "date"])` thì sao? …

## Bài 4 · Đối chiếu cột dẫn xuất với con số tự đếm

Bảng `listings` có sẵn `number_of_reviews` và `number_of_reviews_ltm` (LTM, 12 tháng gần nhất) do Inside Airbnb tính.
Ta tự đếm từ bảng đánh giá để kiểm tra định nghĩa và độ nhất quán.

### Hợp đồng hàm · 15 điểm tự động

`review_count_check(rv: pd.DataFrame, ds: pd.DataFrame) -> dict`

Nhận bảng đánh giá (`listing_id`) và bảng chỗ ở (`id` không trùng, `number_of_reviews`). Trả dict có đúng các khóa:

- `dem_that`: Series int **cùng index với `ds`** — số dòng đánh giá của từng chỗ ở, đếm trên **toàn bộ** `rv`
  (`groupby("listing_id").size()` rồi `map`/`reindex` theo `ds["id"]`), chỗ ở không có đánh giá là 0;
- `so_lech_tong`: int, số chỗ ở có `number_of_reviews` khác `dem_that`.

Không thêm cột vào `ds`. Đối chiếu snapshot: **0** chỗ ở lệch trên 18.534.

In [ ]:
def review_count_check(rv: pd.DataFrame, ds: pd.DataFrame) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q4():
    before = DEMO_DS.copy()
    result = review_count_check(DEMO_RV, DEMO_DS)
    assert set(result) == {"dem_that", "so_lech_tong"}
    dem_that = result["dem_that"]
    assert isinstance(dem_that, pd.Series) and list(dem_that.index) == list(DEMO_DS.index)
    assert dem_that.tolist() == [7, 4, 4, 2, 0, 1], "Chỗ ở 15 không có đánh giá phải là 0, không phải NaN"
    assert result["so_lech_tong"] == 0
    pd.testing.assert_frame_equal(DEMO_DS, before, obj="DEMO_DS (không thêm cột)")

public_check("Q4 · ví dụ công khai", check_q4)

**Khớp toàn bộ nghĩa là gì?** `number_of_reviews` có bao gồm các đánh giá sau mốc 29/06 không? Nếu quy trình cắt tại 29/06
thì cột dẫn xuất phải xử lý thế nào? …

## Bài 5 · Cột LTM: lệch định nghĩa cửa sổ 12 tháng

Thử định nghĩa "12 tháng gần nhất" là các đánh giá có `date >= LTM_BAT_DAU` (snapshot: `"2025-06-30"`), tự đếm rồi so
với `number_of_reviews_ltm`. Xem **phân bố** độ lệch, không chỉ đếm số dòng lệch.

### Hợp đồng hàm · 20 điểm tự động

`ltm_mismatch(rv: pd.DataFrame, ds: pd.DataFrame, bat_dau: str, ket_thuc: str | None = None) -> dict`

Nhận hai bảng (ds có `id`, `number_of_reviews_ltm`), ngày bắt đầu cửa sổ và ngày kết thúc tùy chọn (`None` = không chặn
trên; có giá trị thì gồm cả ngày đó). Trả dict có đúng các khóa:

- `ltm_that`: Series int cùng index với `ds` — số đánh giá có `bat_dau <= date` (và `date <= ket_thuc` nếu có), 0 nếu không có;
- `do_lech`: Series int cùng index với `ds` — `number_of_reviews_ltm − ltm_that`;
- `so_lech_ltm`: int, số chỗ ở có `do_lech` khác 0.

Không sửa input. Đối chiếu snapshot (`bat_dau="2025-06-30"`, không chặn trên): **665** chỗ ở lệch, phần lớn lệch **−1**.

In [ ]:
def ltm_mismatch(rv: pd.DataFrame, ds: pd.DataFrame, bat_dau: str, ket_thuc: str | None = None) -> dict:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q5():
    result = ltm_mismatch(DEMO_RV, DEMO_DS, LTM_BAT_DAU)
    assert set(result) == {"ltm_that", "do_lech", "so_lech_ltm"}
    assert result["ltm_that"].tolist() == [5, 3, 1, 2, 0, 0]
    assert result["do_lech"].tolist() == [-1, -1, 0, 0, 0, 0]
    assert result["so_lech_ltm"] == 2
    chan = ltm_mismatch(DEMO_RV, DEMO_DS, LTM_BAT_DAU, MOC)
    assert chan["ltm_that"].tolist() == [4, 2, 1, 1, 0, 0] and chan["so_lech_ltm"] == 1

public_check("Q5 · ví dụ công khai", check_q5)

In [ ]:
def thuc_hanh_q1_q5():
    global so_lieu
    mien = time_domain(rv, MOC)
    ltm = ltm_mismatch(rv, ds, LTM_BAT_DAU)
    so_lieu = {
        "review_sau_moc": mien["so_sau_moc"],
        "review_mo_coi": orphan_reviews(rv, ds),
        "trung_theo_cap": duplicate_pairs(rv),
        "lech_tong_review": review_count_check(rv, ds)["so_lech_tong"],
        "lech_ltm": ltm["so_lech_ltm"],
    }
    lech = ltm["do_lech"]
    return (f"Ngày đánh giá từ {mien['ngay_dau']:%Y-%m-%d} đến {mien['ngay_cuoi']:%Y-%m-%d}\n{so_lieu}\n"
            f"Độ lệch LTM trong nhóm lệch:\n{lech[lech != 0].describe().round(2)}")

preview("Bài 1–5 trên dữ liệu đang dùng", thuc_hanh_q1_q5)

**Phân bố độ lệch:** độ lệch tập trung ở giá trị nào? Điều đó gợi ý gì về định nghĩa cửa sổ của Inside Airbnb, và vì sao chưa đủ để khẳng định? …

## Bài 6 · Đóng gói `qa_report`

Gom kết quả các phép kiểm thành một bảng ba cột và ghi ra file — sản phẩm theo yêu cầu của bài tập lớn.
Cell thực hành phía trên đã gom số liệu vào dict `so_lieu`.

### Hợp đồng hàm · 15 điểm tự động

`qa_report(so_dong: dict, ghi_chu: dict, path) -> pd.DataFrame`

Nhận dict `{tên quy tắc: số dòng (int)}` (giữ thứ tự khóa), dict `{tên quy tắc: ghi chú}` và đường dẫn file CSV
(thư mục cha đã tồn tại). Tạo DataFrame đúng ba cột `quy_tac`, `so_dong`, `ghi_chu`, **mỗi quy tắc trong `so_dong` một dòng**
theo đúng thứ tự — kể cả quy tắc có 0 dòng; quy tắc không có trong `ghi_chu` thì ghi chú là chuỗi rỗng `""`.
Ghi bằng `to_csv(path, index=False)` (UTF-8), đọc lại bằng `pd.read_csv(path, keep_default_na=False)` và trả bảng đọc lại.
Không sửa input.

In [ ]:
def qa_report(so_dong: dict, ghi_chu: dict, path) -> pd.DataFrame:
    # TODO: triển khai đúng hợp đồng ở trên.
    raise NotImplementedError

In [ ]:
def check_q6():
    import tempfile
    from pathlib import Path
    so = {"review_sau_moc": 3, "review_mo_coi": 0, "trung_theo_cap": 3}
    ghi = {"review_sau_moc": "date > mốc công bố", "trung_theo_cap": "không xoá"}
    with tempfile.TemporaryDirectory() as directory:
        path = Path(directory) / "qa_report.csv"
        kq = qa_report(so, ghi, path)
        assert path.exists(), "Cần thực sự ghi file"
        header = path.read_text(encoding="utf-8").splitlines()[0]
    assert header == "quy_tac,so_dong,ghi_chu", "Kiểm tra index=False và thứ tự cột"
    assert isinstance(kq, pd.DataFrame) and kq.shape == (3, 3)
    assert kq["quy_tac"].tolist() == list(so), "Giữ thứ tự quy tắc, kể cả quy tắc có 0 dòng"
    assert kq.set_index("quy_tac")["so_dong"].to_dict() == so
    assert kq.loc[1, "ghi_chu"] == ""

public_check("Q6 · ví dụ công khai", check_q6)

In [ ]:
GHI_CHU = {
    "review_sau_moc": "date > mốc công bố; áp dụng chính sách cắt mốc đã ghi rõ",
    "review_mo_coi": "listing_id không có trong listings",
    "trung_theo_cap": "không xoá; hai cột không phải khoá tự nhiên",
    "lech_tong_review": "cột dẫn xuất so với số tự đếm",
    "lech_ltm": "khác định nghĩa mốc 12 tháng",
}
preview("Bài 6 — qa_report_reviews.csv", lambda: qa_report(so_lieu, GHI_CHU, "qa_report_reviews.csv"))

## Bài tự làm ✅ mở — không cộng vào 80 điểm tự động

**Bộ QA trên mốc chụp khác.** Chạy lại năm phép kiểm trên mốc **2025-09-27**
(đổi `BASE` thành `.../2025-09-27/visualisations`; cập nhật `MOC` và cửa sổ LTM).
Quy tắc nào cho kết quả khác? Ghi lại thay đổi cần thiết để cùng một bộ quy tắc chạy được
trên một mốc chụp khác (bộ test chung khi chấm có thể dùng mốc như vậy).

In [ ]:
# Viết bài tự làm của bạn ở đây (không được chấm tự động).

**Bài tự làm của tôi:** …

## Khai báo sử dụng AI

- **Công cụ đã dùng:** … *(hoặc “không dùng”)*
- **Bài đã dùng AI và yêu cầu chính:** …
- **Một gợi ý đã kiểm chứng, cách kiểm chứng và kết quả:** …

Không chỉ ghi “code chạy được”; nêu số dòng, ngày hoặc phép đếm tay đã dùng để đối chiếu.

---

## 🧭 Hỗ trợ bài tập lớn (~30 phút — làm việc theo nhóm, không tính điểm lab)

Tự soát theo checklist; đánh dấu ✅/❌ và ghi một dòng "việc tuần tới" cho mỗi ❌:

1. ☐ Repo GitHub **riêng tư** đã mời tài khoản giảng viên và giảng viên thực hành
   (danh sách trên Canvas Portal); cấu trúc thư mục đúng theo đề.
2. ☐ **Mã tải dữ liệu** chạy được: tải đủ các mốc chụp bắt buộc của thành phố nhóm
   vào `data/raw/` — chưa commit dữ liệu thô lên repo.
3. ☐ **Kế hoạch phân tích** (bên đặt hàng, câu hỏi, KPI dự kiến, kế hoạch LLM) đã viết ra,
   cả nhóm thống nhất và chia thành đầu việc có người phụ trách.
4. ☐ **Bộ quy tắc QA nháp** (theo bài 10 + lab này): mỗi quy tắc đủ 4 phần
   tên–điều kiện–lý do–hành động; chạy thử trên mốc chụp mới nhất và tạo `qa_report` đầu tiên.
5. ☐ Phân công tuần tới: ai phụ trách **hợp phần LLM** (bài 11) — người đó tạo API key
   Gemini trước ở nhà (aistudio.google.com, miễn phí).

> Nhóm hoàn thành sớm có thể chạy bộ QA trên **một mốc chụp chưa dùng** để kiểm tra
> khả năng áp dụng trên mốc chụp lạ.

## Tóm tắt bài lab

| Bạn đã làm | Dùng cho |
|---|---|
| Kiểm miền thời gian và khoá ngoại | `qa_report` của bài tập lớn |
| Chứng minh bảng không có khoá tự nhiên | tránh dùng `drop_duplicates` máy móc |
| Đối chiếu cột dẫn xuất với số tự đếm | kiểm tra định nghĩa trước khi sử dụng |
| `qa_report.csv` từ các hàm nhận bảng qua tham số | chạy lại trên mốc chụp khác |

**Trước khi nộp:** hoàn thành TODO và câu trả lời, chạy Restart & Run all, lưu notebook.

Bài giảng tới: **mô hình ngôn ngữ lớn (LLM) cho dữ liệu phi cấu trúc**.
Hãy chuẩn bị API key Gemini theo hướng dẫn ở đầu notebook Bài 11.

## Thông tin và nộp bài

**Họ tên / MSSV / lớp:** [Điền]

1. Giữ tên file **`lab-10.ipynb`** ở gốc repo (dấu gạch ngang, không phải `lab_10.ipynb`).
2. Hoàn thành hàm TODO, các diễn giải và khai báo AI.
3. Restart & Run all; đọc bảng public checks, lưu file rồi commit/push lên `main`.
   Không commit `qa_report_*.csv` do notebook tạo ra.
4. Tạo tag `submit/lab-10/v1`; lần sửa tiếp dùng v2 hoặc v3, không di chuyển tag đã nộp.
5. Repo cần được giảng viên đăng ký chấm. Xem status `private-grader/lab-10` trên commit nộp;
   điểm tự động tối đa 80, còn 20 điểm chờ giảng viên. Xem lịch quét trong README.

Tag v1/v2/v3 là **lần nộp**, không phải phiên bản đề. Không đổi metadata để chọn bộ test khác.

In [ ]:
show_public_summary()